In [5]:
import pandas as pd
import numpy as np

# Load the Telco dataset
df = pd.read_csv('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')

In [6]:
# Rename existing columns to match the project contract
df = df.rename(columns={
    'customerID': 'user_id',
    'MonthlyCharges': 'monthly_spend'
})

# Synthesize missing date columns using a mock observation date
mock_today = pd.to_datetime('2024-01-01')

# Convert existing tenure (months) to approximate days
tenure_days = df['tenure'] * 30.436875

# Back-calculate signup_date
df['signup_date'] = mock_today - pd.to_timedelta(tenure_days, unit='D')

# Set cancel_date only for users who churned; active users remain NaT (null)
df['cancel_date'] = np.where(df['Churn'] == 'Yes', mock_today, pd.NaT)

# Mock last active date for RFM calculation later
df['last_active_date'] = mock_today

# Drop original tenure so we are forced to calculate it via datetime math
df = df.drop(columns=['tenure'])

print("Mapped columns ready for ETL:", df.columns.tolist())

Mapped columns ready for ETL: ['user_id', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'monthly_spend', 'TotalCharges', 'Churn', 'signup_date', 'cancel_date', 'last_active_date']


In [7]:
# Convert string/synthesized columns to explicit datetime objects
date_columns = ['signup_date', 'cancel_date', 'last_active_date']
for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors='coerce')

In [8]:
# Establish "today" as the maximum observation date present in any date column
observation_date = df[['signup_date', 'cancel_date', 'last_active_date']].max().max()

# Create binary churn flag: 1 if cancel_date exists, 0 if null (active/censored)
df['churned'] = df['cancel_date'].notnull().astype(int)

# Create an effective end date: actual cancel_date OR observation_date for active users
df['effective_end_date'] = df['cancel_date'].fillna(observation_date)

In [9]:
# Calculate raw timedelta in days, accounting for leap years[cite: 1]
df['tenure_days'] = (df['effective_end_date'] - df['signup_date']).dt.days

# Convert to months using the exact Gregorian calendar average (365.2425 days / 12)
df['tenure_months'] = df['tenure_days'] / 30.436875

In [10]:
print("--- Active (0) vs Churned (1) ---")
print(df['churned'].value_counts())

print("\n--- Tenure Summary (Months) ---")
print(df['tenure_months'].describe())

# Check specific leap year edge cases
leap_year_signups = df[df['signup_date'].dt.date == pd.to_datetime('2024-02-29').date()]
if not leap_year_signups.empty:
    print("\n--- Leap Year Signups (Feb 29, 2024) ---")
    display(leap_year_signups[['signup_date', 'effective_end_date', 'tenure_days', 'tenure_months']].head())

--- Active (0) vs Churned (1) ---
churned
0    5174
1    1869
Name: count, dtype: int64

--- Tenure Summary (Months) ---
count    7043.000000
mean       32.354891
std        24.560320
min         0.000000
25%         8.969383
50%        28.978008
75%        54.999076
max        71.985051
Name: tenure_months, dtype: float64
